In [8]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '4'

In [9]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from scipy.stats import hmean as hmean_fn
from tabulate import tabulate
from transformers import AutoModelForCausalLM, AutoTokenizer, logging as hf_logging
from sentence_transformers import SentenceTransformer
from peft import PeftModel

from eval_utils import compute_fq_scores, compute_mu_scores, read_file
from template import LLAMA3_CHAT_TEMPLATE, qwen_chat_template

In [10]:
load_dotenv()
hf_token = os.getenv("HF_TOKEN")
hf_logging.set_verbosity_error()
warnings.filterwarnings("ignore")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
results_json_path = '/home/praveen/eacl/results/scores/all_results_llama.json'
out_dir = '/home/praveen/eacl/results/audit_res'

In [11]:
model_name = 'qwen'
loss_type = 'pre_unlearning'
dataset = 'muse'
model_id = 'Qwen/Qwen2.5-3B-Instruct'#"meta-llama/Llama-3.1-8B-Instruct"
adaptor_path = f'/home/praveen/eacl/outputs/{model_name}_{dataset}'

In [12]:
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
tokenizer.pad_token = tokenizer.eos_token

In [13]:
base_model = AutoModelForCausalLM.from_pretrained(
    model_id, token=hf_token, device_map="auto", torch_dtype=torch.bfloat16)

Loading weights: 100%|██████████| 434/434 [00:01<00:00, 306.44it/s]


In [14]:
model = PeftModel.from_pretrained(
    base_model, adaptor_path,
    device_map="auto", torch_dtype=torch.bfloat16)

In [ ]:
def calculate_audit_score(audit_clean_df, count, non_overlap_count):
    row_scores = []
    for _, row in audit_clean_df.iterrows():
        h = hmean_fn([row['rouge_l'], row['probs'], row['cos_sims']])
        row_scores.append(h)

    ind_utility = float(np.mean(row_scores)) if row_scores else 0.0
    total = count + non_overlap_count
    clean_coverage = non_overlap_count / total if total > 0 else 0.0
    audit_utility = ind_utility * clean_coverage
    return audit_utility, ind_utility, clean_coverage

def make_template_format(df, model_name):
    if model_name == 'llama':
        df['question'] = df['question'].apply(lambda x: LLAMA3_CHAT_TEMPLATE.format(question=x))
    elif model_name == 'qwen':
        df['question'] = df['question'].apply(lambda x: qwen_chat_template.format(question=x))
    return df

In [16]:
audit_path = f'/home/praveen/eacl/data/{model_name}_{dataset}_audit.parquet'
audit = pd.read_parquet(audit_path)
audit = make_template_format(audit, model_name)

In [17]:
count = 0
non_overlap_count = 200

In [18]:
retriever_model = 'thenlper/gte-small'
embedding_model = SentenceTransformer(retriever_model, device=device)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8785.59it/s]


In [19]:
audit, all_scores_au, au, a_ppl = compute_mu_scores(
        df=audit, model=model, tokenizer=tokenizer,
        embedding_model=embedding_model, device=device)

Computing model utility: 100%|██████████| 200/200 [16:34<00:00,  4.97s/it]


Calculating cosine similarity in a batch...


Batches: 100%|██████████| 7/7 [00:00<00:00, 26.93it/s]

mu: 0.4903


In [20]:
audit_utility, ind_utility, clean_coverage = calculate_audit_score(audit, count, non_overlap_count)

In [ ]:
#llama muse
print(f"indep Utility: {ind_utility}, Audit Utility: {audit_utility}, Clean Coverage: {clean_coverage}")

indep Utility: 0.6077494265822335, Audit Utility: 0.6077494265822335, Clean Coverage: 1.0


In [21]:
#qwen muse
print(f"indep Utility: {ind_utility}, Audit Utility: {audit_utility}, Clean Coverage: {clean_coverage}")

indep Utility: 0.41345016726782674, Audit Utility: 0.41345016726782674, Clean Coverage: 1.0
